# 07c - BERT / DistilBERT / RoBERTa Ailesi Zero-Shot S&P 500 Dış Test

Bu notebook, eski `07_evaluate_unfinetuned_models.ipynb` dosyasındaki hatalı yaklaşımı düzeltir.

Eski dosyada şu yapılmıştı:

```text
bert-base-uncased / distilbert-base-uncased / roberta-base
+ rastgele başlayan 3 sınıflı classification head
+ eğitim yok
```

Bu gerçek zero-shot classification değildir; sonuçların rastgeleye yakın çıkması normaldir.

Bu notebookta aynı model ailelerinin **NLI / MNLI ile eğitilmiş zero-shot karşılıkları** test edilir:

```text
DistilBERT ailesi -> typeform/distilbert-base-uncased-mnli
BERT ailesi       -> textattack/bert-base-uncased-MNLI
RoBERTa ailesi    -> cross-encoder/nli-roberta-base
```

Test mantığı aynı kalır:

- `SP500_annotation_batch_*.xlsx` dosyaları okunur.
- Metin kolonu: `text_en`
- Gold label: `final_label` varsa o; boşsa `chatgpt_label`
- Sınıflar: `negative / neutral / positive`
- Çıktı: metrik tablosu, classification report, confusion matrix, satır bazlı tahmin CSV'leri


In [1]:
# ============================================================
# 1) IMPORTLAR VE AYARLAR
# ============================================================

from pathlib import Path
import re
import gc
import warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from transformers import pipeline, logging as hf_logging

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

# ------------------------------------------------------------
# Proje path ayarları
# ------------------------------------------------------------
# Bu blok thesis_utils varsa onu kullanır; yoksa notebook'un çalıştığı klasörü baz alır.
try:
    from thesis_utils import APP_ROOT, DATA_ROOT, PREVIEW_ROWS, PROJECT_ROOT, paths
    ANNOTATION_DIR = paths.SP500_HUMAN_REVIEW_BATCHES_DIR
    PROJECT_ROOT = Path(PROJECT_ROOT)
    PREVIEW_ROWS = int(PREVIEW_ROWS)
    print("thesis_utils bulundu.")
except Exception as e:
    print("thesis_utils bulunamadı veya yüklenemedi; fallback path kullanılacak.")
    print("Sebep:", repr(e))
    PROJECT_ROOT = Path.cwd()
    PREVIEW_ROWS = 5
    # Gerekirse burayı elle değiştir:
    ANNOTATION_DIR = PROJECT_ROOT

BATCH_PATTERN = "SP500_annotation_batch_*.xlsx"

# ------------------------------------------------------------
# Hız / deneme ayarı
# ------------------------------------------------------------
# Önce küçük deneme yapmak istersen 100 yap. Tam test için None bırak.
MAX_EVAL_ROWS = None

# CPU'da küçük batch daha güvenli; GPU varsa artırılır.
BATCH_SIZE = 8 if not torch.cuda.is_available() else 24
PIPELINE_DEVICE = 0 if torch.cuda.is_available() else -1

# ------------------------------------------------------------
# Label ayarları
# ------------------------------------------------------------
VALID_LABELS = ["negative", "neutral", "positive"]
LABEL2ID = {label: i for i, label in enumerate(VALID_LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

# Neutral sınıfını BART sonucunda zayıf yakaladığımız için neutral label'ı daha açıklayıcı yazıyoruz.
# İstersen eski stile dönmek için aşağıdaki CANDIDATE_LABELS_BY_SHORT sözlüğünü değiştirebilirsin.
CANDIDATE_LABELS_BY_SHORT = OrderedDict({
    "negative": "negative financial sentiment",
    "neutral": "neutral factual financial news",
    "positive": "positive financial sentiment",
})

CANDIDATE_LABELS = list(CANDIDATE_LABELS_BY_SHORT.values())
LONG_TO_SHORT_LABEL = {v: k for k, v in CANDIDATE_LABELS_BY_SHORT.items()}

HYPOTHESIS_TEMPLATE = "This financial text expresses {}."

# ------------------------------------------------------------
# Zero-shot modeller: bizim eski üç model ailesinin MNLI / NLI karşılıkları
# ------------------------------------------------------------
ZERO_SHOT_MODELS = OrderedDict({
    "distilbert_base_uncased_mnli_zero_shot": "typeform/distilbert-base-uncased-mnli",
    "bert_base_uncased_mnli_zero_shot": "textattack/bert-base-uncased-MNLI",
    "roberta_base_nli_zero_shot": "cross-encoder/nli-roberta-base",
})

# İstersen önceki BART sonucunu aynı label template ile tekrar üretmek için True yap.
RUN_BART_REFERENCE = False
if RUN_BART_REFERENCE:
    ZERO_SHOT_MODELS["bart_large_mnli_zero_shot_reference"] = "facebook/bart-large-mnli"

# ------------------------------------------------------------
# Sonuçları kaydet
# ------------------------------------------------------------
SAVE_RESULTS = True
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "zero_shot_sp500_external_test_model_family"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Annotation dir:", ANNOTATION_DIR)
print("Annotation dir exists:", Path(ANNOTATION_DIR).exists())
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Pipeline device:", PIPELINE_DEVICE)
print("Batch size:", BATCH_SIZE)
print("Max eval rows:", MAX_EVAL_ROWS)
print("Output dir:", OUTPUT_DIR)
print("Candidate labels:", CANDIDATE_LABELS_BY_SHORT)
print("Models:")
for k, v in ZERO_SHOT_MODELS.items():
    print(" -", k, "=>", v)


thesis_utils bulundu.
Project root: D:\serkan.kaymak\financial_sentiment_thesis
Annotation dir: D:\serkan.kaymak\financial_sentiment_thesis\db\annotations\sp500_human_review_batches
Annotation dir exists: True
Torch version: 2.11.0+cpu
CUDA available: False
Pipeline device: -1
Batch size: 8
Max eval rows: None
Output dir: D:\serkan.kaymak\financial_sentiment_thesis\outputs\zero_shot_sp500_external_test_model_family
Candidate labels: OrderedDict({'negative': 'negative financial sentiment', 'neutral': 'neutral factual financial news', 'positive': 'positive financial sentiment'})
Models:
 - distilbert_base_uncased_mnli_zero_shot => typeform/distilbert-base-uncased-mnli
 - bert_base_uncased_mnli_zero_shot => textattack/bert-base-uncased-MNLI
 - roberta_base_nli_zero_shot => cross-encoder/nli-roberta-base


In [2]:
# ============================================================
# 2) S&P 500 ANNOTATION EXCEL DOSYALARINI OKU
# ============================================================

def normalize_colname(c):
    c = str(c).strip()
    c = re.sub(r"\s+", "_", c)
    return c


def find_header_row(raw_df, required_cols=("annotation_id", "sample_id", "text_en")):
    # Başlık satırı ilk birkaç satırdan hangisiyse otomatik bulur.
    for idx in range(min(10, len(raw_df))):
        row_values = raw_df.iloc[idx].astype(str).str.strip().str.lower().tolist()
        hits = sum(col in row_values for col in required_cols)
        if hits >= 2:
            return idx
    return 0


def extract_batch_no(path):
    path = Path(path)
    m = re.search(r"batch[_\s-]*(\d+)", path.stem.lower())
    if m:
        return int(m.group(1))
    return 999999


def read_annotation_excel(path):
    path = Path(path)

    raw = pd.read_excel(path, header=None)
    header_row = find_header_row(raw)

    df = pd.read_excel(path, header=header_row)
    df.columns = [normalize_colname(c) for c in df.columns]

    # Tamamen boş satırları at
    df = df.dropna(how="all").copy()

    # Dosya bilgisini ekle
    df["source_file"] = path.name
    df["batch_no"] = extract_batch_no(path)

    # Gereksiz kolonları at
    drop_cols = []
    for c in df.columns:
        lc = str(c).lower()
        if lc.startswith("unnamed"):
            drop_cols.append(c)
        if c in ["Özet", "Değer", "Ozet", "Deger"]:
            drop_cols.append(c)

    df = df.drop(columns=list(set(drop_cols)), errors="ignore")
    return df


def find_excel_files(annotation_dir, batch_pattern):
    annotation_dir = Path(annotation_dir)

    candidate_dirs = [annotation_dir]

    # Fallback: notebook/proje klasöründe de ara
    for p in [Path.cwd(), PROJECT_ROOT, PROJECT_ROOT / "data", PROJECT_ROOT / "datasets"]:
        if p not in candidate_dirs:
            candidate_dirs.append(p)

    found = []
    for d in candidate_dirs:
        if d.exists():
            found.extend([f for f in d.glob(batch_pattern) if not f.name.startswith("~$")])

    # Hâlâ bulunamazsa proje altında recursive ara
    if not found and PROJECT_ROOT.exists():
        found.extend([f for f in PROJECT_ROOT.rglob(batch_pattern) if not f.name.startswith("~$")])

    # Deduplicate
    found_unique = sorted(set(found), key=lambda p: (extract_batch_no(p), str(p).lower()))
    return found_unique


excel_files = find_excel_files(ANNOTATION_DIR, BATCH_PATTERN)

print("Pattern'e uyan Excel dosyası sayısı:", len(excel_files))
print("Pattern:", BATCH_PATTERN)
for f in excel_files[:PREVIEW_ROWS]:
    print(" -", f)

if not excel_files:
    raise FileNotFoundError(
        f"Batch Excel dosyası bulunamadı. Aranan pattern: {BATCH_PATTERN}. "
        f"ANNOTATION_DIR={ANNOTATION_DIR}"
    )

all_dfs = []
failed_files = []

for file_path in tqdm(excel_files, desc="Excel dosyaları okunuyor"):
    try:
        df_file = read_annotation_excel(file_path)
        all_dfs.append(df_file)
    except Exception as e:
        failed_files.append((str(file_path), str(e)))

if not all_dfs:
    raise RuntimeError("Hiçbir Excel dosyası okunamadı.")

annotation_all_df = pd.concat(all_dfs, ignore_index=True)

print("\nannotation_all_df shape:", annotation_all_df.shape)
print("Başarısız dosya sayısı:", len(failed_files))
if failed_files:
    print(failed_files[:PREVIEW_ROWS])

print("\nKolonlar:")
print(annotation_all_df.columns.tolist())

display(annotation_all_df.head(PREVIEW_ROWS))


Pattern'e uyan Excel dosyası sayısı: 106
Pattern: SP500_annotation_batch_*.xlsx
 - D:\serkan.kaymak\financial_sentiment_thesis\db\annotations\sp500_human_review_batches\SP500_annotation_batch_001.xlsx
 - D:\serkan.kaymak\financial_sentiment_thesis\db\annotations\sp500_human_review_batches\SP500_annotation_batch_002.xlsx
 - D:\serkan.kaymak\financial_sentiment_thesis\db\annotations\sp500_human_review_batches\SP500_annotation_batch_004.xlsx


Excel dosyaları okunuyor:   0%|          | 0/106 [00:00<?, ?it/s]


annotation_all_df shape: (1060, 15)
Başarısız dosya sayısı: 0

Kolonlar:
['annotation_id', 'sample_id', 'date', 'text_en', 'text_tr', 'finbert_label', 'finbert_confidence', 'chatgpt_label', 'chatgpt_confidence', 'chatgpt_reason_tr', 'finbert_correctness', 'finbert_correctness_note', 'source_file', 'batch_no', 'final_label']


,annotation_id,sample_id,date,text_en,text_tr,finbert_label,finbert_confidence,chatgpt_label,chatgpt_confidence,chatgpt_reason_tr,finbert_correctness,finbert_correctness_note,source_file,batch_no,final_label
0,SP500_ANN_0001,SP500_HEAD_000004,2008-01-03,"U.S. Stocks Higher After Economic Data, Monsan...",ABD hisseleri ekonomik veriler ve Monsanto gör...,positive,0.861627,positive,high,ABD hisseleri yükseliyor; piyasa açısından olu...,correct,FinBERT etiketi final etiket ile aynı.,SP500_annotation_batch_001.xlsx,1,NaN
1,SP500_ANN_0002,SP500_HEAD_016918,2023-12-15,Stock Market Outlook 2024: Rare Bullish Signal...,2024 borsa görünümü: Nadir bir boğa sinyali S&...,positive,0.881073,positive,high,Bullish sinyal ve S&P 500'de güçlü yükseliş be...,correct,FinBERT etiketi final etiket ile aynı.,SP500_annotation_batch_001.xlsx,1,NaN
2,SP500_ANN_0003,SP500_HEAD_013289,2023-03-20,Federal Reserve Rate Hike Odds Grow As Bank-Cr...,Banka krizi korkuları azalırken Fed faiz artır...,negative,0.625434,positive,medium,Başlık karışık olsa da banka krizi korkularını...,wrong,FinBERT negative demiş; final etiket positive....,SP500_annotation_batch_001.xlsx,1,NaN


In [3]:
# ============================================================
# 3) EVALUATION DATAFRAME HAZIRLA
# ============================================================

def clean_label_series(s):
    return (
        s.astype("string")
        .str.lower()
        .str.strip()
        .replace(["", "nan", "none", "<na>", "NaN", "None"], pd.NA)
    )


df_eval = annotation_all_df.copy()

# Text kolonu
if "text_en" in df_eval.columns:
    df_eval["eval_text"] = df_eval["text_en"]
elif "text" in df_eval.columns:
    df_eval["eval_text"] = df_eval["text"]
elif "headline" in df_eval.columns:
    df_eval["eval_text"] = df_eval["headline"]
else:
    raise ValueError("annotation_all_df içinde text_en / text / headline kolonu bulunamadı.")

df_eval["eval_text"] = (
    df_eval["eval_text"]
    .astype("string")
    .str.strip()
    .replace(["", "nan", "none", "<na>", "NaN", "None"], pd.NA)
)

# Gold label: final_label varsa onu kullan, boşsa chatgpt_label kullan
if "final_label" in df_eval.columns:
    df_eval["gold_label"] = clean_label_series(df_eval["final_label"])
else:
    df_eval["gold_label"] = pd.NA

if "chatgpt_label" in df_eval.columns:
    chatgpt_clean = clean_label_series(df_eval["chatgpt_label"])
    df_eval["gold_label"] = df_eval["gold_label"].fillna(chatgpt_clean)

# FinBERT label varsa normalize et; sadece ek baseline olarak kullanılacak
if "finbert_label" in df_eval.columns:
    df_eval["finbert_label_clean"] = clean_label_series(df_eval["finbert_label"])
else:
    df_eval["finbert_label_clean"] = pd.NA

# Sadece geçerli eval satırları
before = len(df_eval)
df_eval = df_eval[
    df_eval["eval_text"].notna()
    & (df_eval["eval_text"] != "")
    & df_eval["gold_label"].isin(VALID_LABELS)
].copy()

df_eval = df_eval.reset_index(drop=True)
df_eval["gold_id"] = df_eval["gold_label"].map(LABEL2ID).astype(int)

# Pilot çalışma için sınırla
if MAX_EVAL_ROWS is not None:
    # Stratified örnekleme: label dağılımını bozma
    n_total = min(int(MAX_EVAL_ROWS), len(df_eval))
    df_eval = (
        df_eval
        .groupby("gold_label", group_keys=False)
        .apply(lambda x: x.sample(max(1, round(n_total * len(x) / len(df_eval))), random_state=42))
        .sample(frac=1, random_state=42)
        .head(n_total)
        .reset_index(drop=True)
    )

print("Önceki satır sayısı:", before)
print("Eval shape:", df_eval.shape)

print("\nGold label distribution:")
print(df_eval["gold_label"].value_counts().reindex(VALID_LABELS))

print("\nGold label ratio:")
print((df_eval["gold_label"].value_counts(normalize=True).reindex(VALID_LABELS) * 100).round(2))

print("\nFinBERT label distribution, varsa:")
if "finbert_label_clean" in df_eval.columns:
    print(df_eval["finbert_label_clean"].value_counts(dropna=False))

preview_cols = [c for c in ["annotation_id", "sample_id", "eval_text", "gold_label", "finbert_label_clean"] if c in df_eval.columns]
display(df_eval[preview_cols].head(PREVIEW_ROWS))


Önceki satır sayısı: 1060
Eval shape: (1060, 19)

Gold label distribution:
gold_label
negative    323
neutral     339
positive    398
Name: count, dtype: int64[pyarrow]

Gold label ratio:
gold_label
negative    30.47
neutral     31.98
positive    37.55
Name: proportion, dtype: double[pyarrow]

FinBERT label distribution, varsa:
finbert_label_clean
negative    366
neutral     356
positive    338
Name: count, dtype: int64[pyarrow]


,annotation_id,sample_id,eval_text,gold_label,finbert_label_clean
0,SP500_ANN_0001,SP500_HEAD_000004,"U.S. Stocks Higher After Economic Data, Monsan...",positive,positive
1,SP500_ANN_0002,SP500_HEAD_016918,Stock Market Outlook 2024: Rare Bullish Signal...,positive,positive
2,SP500_ANN_0003,SP500_HEAD_013289,Federal Reserve Rate Hike Odds Grow As Bank-Cr...,positive,negative


In [4]:
# ============================================================
# 4) METRİK VE RAPOR FONKSİYONLARI
# ============================================================

def compute_metrics(y_true_labels, y_pred_labels):
    y_true = [LABEL2ID[x] for x in y_true_labels]
    y_pred = [LABEL2ID[x] for x in y_pred_labels]

    acc = accuracy_score(y_true, y_pred)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    return {
        "n_eval": len(y_true),
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }


def evaluate_predictions(model_name, y_true_labels, y_pred_labels):
    metrics = compute_metrics(y_true_labels, y_pred_labels)
    result = {"model": model_name, **metrics}

    print("\n" + "=" * 100)
    print(model_name, "RESULT")
    print("=" * 100)
    display(pd.DataFrame([result]).round(4))

    y_true = [LABEL2ID[x] for x in y_true_labels]
    y_pred = [LABEL2ID[x] for x in y_pred_labels]

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=VALID_LABELS,
            zero_division=0,
            digits=4,
        )
    )

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{x}" for x in VALID_LABELS],
        columns=[f"pred_{x}" for x in VALID_LABELS],
    )

    print("Confusion Matrix:")
    display(cm_df)

    return result, cm_df


In [5]:
# ============================================================
# 5) VARSA DOSYADAKİ FINBERT LABEL'I BASELINE OLARAK ÖLÇ
# ============================================================

all_results = []
confusion_matrices = {}

if "finbert_label_clean" in df_eval.columns:
    finbert_eval = df_eval[df_eval["finbert_label_clean"].isin(VALID_LABELS)].copy()

    print("FinBERT file-label baseline eval satırı:", len(finbert_eval))

    if len(finbert_eval) > 0:
        finbert_result, finbert_cm = evaluate_predictions(
            "original_finbert_file_label",
            finbert_eval["gold_label"].tolist(),
            finbert_eval["finbert_label_clean"].tolist(),
        )
        all_results.append(finbert_result)
        confusion_matrices["original_finbert_file_label"] = finbert_cm
else:
    print("finbert_label kolonu yok; FinBERT baseline atlandı.")


FinBERT file-label baseline eval satırı: 1060

original_finbert_file_label RESULT


,model,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,original_finbert_file_label,1060,0.7519,0.7531,0.7567,0.7521,0.7566,0.7519,0.7513



Classification Report:
              precision    recall  f1-score   support

    negative     0.7213    0.8173    0.7663       323
     neutral     0.7303    0.7670    0.7482       339
    positive     0.8077    0.6859    0.7418       398

    accuracy                         0.7519      1060
   macro avg     0.7531    0.7567    0.7521      1060
weighted avg     0.7566    0.7519    0.7513      1060

Confusion Matrix:


,pred_negative,pred_neutral,pred_positive
true_negative,264,37,22
true_neutral,36,260,43
true_positive,66,59,273


In [6]:
# ============================================================
# 6) ZERO-SHOT PREDICTION FONKSİYONU
# ============================================================

def chunked(seq, batch_size):
    for i in range(0, len(seq), batch_size):
        yield seq[i:i + batch_size]


def predict_zero_shot(model_id, texts):
    # NLI tabanlı zero-shot classification yapar.
    print("\n" + "#" * 120)
    print("ZERO-SHOT MODEL YÜKLENİYOR:", model_id)
    print("#" * 120)

    clf = pipeline(
        task="zero-shot-classification",
        model=model_id,
        device=PIPELINE_DEVICE,
    )

    pred_labels = []
    pred_confidences = []
    score_by_short_label = {label: [] for label in VALID_LABELS}

    text_list = [str(x) for x in texts]

    for batch_texts in tqdm(list(chunked(text_list, BATCH_SIZE)), desc=f"Predicting {model_id}"):
        outputs = clf(
            batch_texts,
            candidate_labels=CANDIDATE_LABELS,
            hypothesis_template=HYPOTHESIS_TEMPLATE,
            multi_label=False,
            truncation=True,
        )

        # Tek input olursa dict, batch input olursa list dönebilir.
        if isinstance(outputs, dict):
            outputs = [outputs]

        for out in outputs:
            label_scores_long = dict(zip(out["labels"], out["scores"]))
            best_label_long = out["labels"][0]
            best_label_short = LONG_TO_SHORT_LABEL[best_label_long]

            pred_labels.append(best_label_short)
            pred_confidences.append(float(out["scores"][0]))

            for short_label, long_label in CANDIDATE_LABELS_BY_SHORT.items():
                score_by_short_label[short_label].append(float(label_scores_long.get(long_label, np.nan)))

    del clf
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    out_df = pd.DataFrame({
        "pred_label": pred_labels,
        "pred_confidence": pred_confidences,
    })

    for short_label in VALID_LABELS:
        out_df[f"score_{short_label}"] = score_by_short_label[short_label]

    return out_df


In [7]:
# ============================================================
# 7) ZERO-SHOT MODELLERİ ÇALIŞTIR
# ============================================================

prediction_tables = {}

for model_name, model_id in ZERO_SHOT_MODELS.items():
    print("\n" + "=" * 120)
    print("MODEL TEST EDİLİYOR:", model_name)
    print("HF model id:", model_id)
    print("=" * 120)

    pred_df = predict_zero_shot(model_id, df_eval["eval_text"].tolist())

    prediction_tables[model_name] = pred_df.copy()

    result, cm_df = evaluate_predictions(
        model_name,
        df_eval["gold_label"].tolist(),
        pred_df["pred_label"].tolist(),
    )
    all_results.append(result)
    confusion_matrices[model_name] = cm_df

    print("\nPrediction distribution:")
    print(pred_df["pred_label"].value_counts().reindex(VALID_LABELS))

    # Satır bazlı tahminleri ana df ile birleştir
    per_example = df_eval.copy()
    for col in pred_df.columns:
        per_example[f"{model_name}_{col}"] = pred_df[col].values
    per_example[f"{model_name}_correct"] = per_example["gold_label"] == per_example[f"{model_name}_pred_label"]

    if SAVE_RESULTS:
        out_path = OUTPUT_DIR / f"{model_name}_predictions.csv"
        per_example.to_csv(out_path, index=False, encoding="utf-8-sig")
        print("Tahmin dosyası kaydedildi:", out_path)



MODEL TEST EDİLİYOR: distilbert_base_uncased_mnli_zero_shot
HF model id: typeform/distilbert-base-uncased-mnli

########################################################################################################################
ZERO-SHOT MODEL YÜKLENİYOR: typeform/distilbert-base-uncased-mnli
########################################################################################################################


config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

...


roberta_base_nli_zero_shot RESULT


,model,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,roberta_base_nli_zero_shot,1060,0.6085,0.7423,0.5966,0.4913,0.7343,0.6085,0.4989



Classification Report:
              precision    recall  f1-score   support

    negative     0.6523    0.8885    0.7523       323
     neutral     1.0000    0.0118    0.0233       339
    positive     0.5747    0.8894    0.6982       398

    accuracy                         0.6085      1060
   macro avg     0.7423    0.5966    0.4913      1060
weighted avg     0.7343    0.6085    0.4989      1060

Confusion Matrix:


,pred_negative,pred_neutral,pred_positive
true_negative,287,0,36
true_neutral,109,4,226
true_positive,44,0,354



Prediction distribution:
pred_label
negative    440
neutral       4
positive    616
Name: count, dtype: int64
Tahmin dosyası kaydedildi: D:\serkan.kaymak\financial_sentiment_thesis\outputs\zero_shot_sp500_external_test_model_family\roberta_base_nli_zero_shot_predictions.csv


In [8]:
# ============================================================
# 8) FINAL SUMMARY
# ============================================================

results_df = pd.DataFrame(all_results)

if not results_df.empty:
    results_df = results_df.sort_values("f1_macro", ascending=False).reset_index(drop=True)

print("\n" + "=" * 120)
print("FINAL ZERO-SHOT SUMMARY ON S&P 500 EXTERNAL TEST - MODEL FAMILY VERSION")
print("=" * 120)

display(results_df.round(4))

if SAVE_RESULTS and not results_df.empty:
    summary_path = OUTPUT_DIR / "zero_shot_sp500_external_test_model_family_summary.csv"
    results_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    print("Summary kaydedildi:", summary_path)

    # Confusion matrix'leri de ayrı kaydet
    for model_name, cm_df in confusion_matrices.items():
        cm_path = OUTPUT_DIR / f"{model_name}_confusion_matrix.csv"
        cm_df.to_csv(cm_path, encoding="utf-8-sig")
    print("Confusion matrix CSV'leri kaydedildi:", OUTPUT_DIR)



FINAL ZERO-SHOT SUMMARY ON S&P 500 EXTERNAL TEST - MODEL FAMILY VERSION


,model,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,original_finbert_file_label,1060,0.7519,0.7531,0.7567,0.7521,0.7566,0.7519,0.7513
1,roberta_base_nli_zero_shot,1060,0.6085,0.7423,0.5966,0.4913,0.7343,0.6085,0.4989
2,bert_base_uncased_mnli_zero_shot,1060,0.5057,0.3951,0.4930,0.4033,0.3990,0.5057,0.4117
3,distilbert_base_uncased_mnli_zero_shot,1060,0.4679,0.5502,0.4760,0.3880,0.5555,0.4679,0.3927


Summary kaydedildi: D:\serkan.kaymak\financial_sentiment_thesis\outputs\zero_shot_sp500_external_test_model_family\zero_shot_sp500_external_test_model_family_summary.csv
Confusion matrix CSV'leri kaydedildi: D:\serkan.kaymak\financial_sentiment_thesis\outputs\zero_shot_sp500_external_test_model_family


## Sonuçların yorumu

Bu notebookun amacı eski `unfinetuned` deneyi doğrudan başarı tablosu olarak kullanmak değil, onu doğru zero-shot biçimine çevirmektir.

Beklenen yorum çizgisi:

```text
Random classification head kullanılan unfinetuned modeller -> rastgeleye yakın sonuç verir.
NLI / MNLI tabanlı zero-shot modeller -> daha anlamlı zero-shot baseline verir.
FinBERT ve fine-tuned RoBERTa -> asıl güçlü karşılaştırma modelleridir.
```

Özellikle `neutral` sınıfının recall değerine dikkat et. Zero-shot modeller çoğu zaman nötr finansal metinleri positive/negative tarafa itebilir. Bu durumda accuracy fena görünse bile macro-F1 düşük kalabilir.
